In [2]:
# Phase 4.4 — NHANES Baseline Model Training
# BalAarogya Vitamin-D Risk Assessment

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, KFold
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
)

# Project root
PROJECT_ROOT = Path.cwd().resolve().parent

# Dataset
DATASET_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "vitamin_d_nhanes_2017_2018.csv"
)

print("Project root:", PROJECT_ROOT)
print("Dataset path:", DATASET_PATH)

Project root: C:\Users\prath\Desktop\BioMed Bharat\BalAarogya-VitaminD
Dataset path: C:\Users\prath\Desktop\BioMed Bharat\BalAarogya-VitaminD\data\processed\vitamin_d_nhanes_2017_2018.csv


In [3]:
# Load analytical dataset

df = pd.read_csv(DATASET_PATH)

print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

print("\nFirst 5 rows:")
display(df.head())

Shape: (702, 7)

Columns:
['SEQN', 'RIDAGEYR', 'RIAGENDR', 'LBXVIDMS', 'BMXWT', 'DBQ197', 'DR1TVD']

First 5 rows:


,SEQN,RIDAGEYR,RIAGENDR,LBXVIDMS,BMXWT,DBQ197,DR1TVD
0,93704.0,2.0,1.0,74.4,13.9,2.0,8.1
1,93720.0,1.0,2.0,62.7,10.6,2.0,0.5
2,93724.0,5.0,2.0,46.2,18.2,3.0,0.0
3,93739.0,5.0,2.0,53.0,22.5,3.0,3.4
4,93769.0,6.0,1.0,48.1,23.5,3.0,NaN


In [4]:
# Basic dataset verification

print("Missing values:")
display(df.isna().sum())

print("\nData types:")
display(df.dtypes)

print("\nTarget summary:")
display(df["LBXVIDMS"].describe())

Missing values:


SEQN          0
RIDAGEYR      0
RIAGENDR      0
LBXVIDMS      0
BMXWT        11
DBQ197        0
DR1TVD      135
dtype: int64


Data types:


SEQN        float64
RIDAGEYR    float64
RIAGENDR    float64
LBXVIDMS    float64
BMXWT       float64
DBQ197      float64
DR1TVD      float64
dtype: object


Target summary:


count    702.000000
mean      71.976781
std       19.489841
min       14.100000
25%       59.125000
50%       71.300000
75%       84.600000
max      146.000000
Name: LBXVIDMS, dtype: float64

In [5]:
# Define predictors (X) and target (y)

TARGET = "LBXVIDMS"
ID_COLUMN = "SEQN"

FEATURES = [
    "RIDAGEYR",
    "RIAGENDR",
    "BMXWT",
    "DBQ197",
    "DR1TVD",
]

X = df[FEATURES].copy()
y = df[TARGET].copy()

print("X shape:", X.shape)
print("y shape:", y.shape)

print("\nPredictors:")
print(X.columns.tolist())

print("\nTarget:")
print(TARGET)

X shape: (702, 5)
y shape: (702,)

Predictors:
['RIDAGEYR', 'RIAGENDR', 'BMXWT', 'DBQ197', 'DR1TVD']

Target:
LBXVIDMS


In [6]:
# Basic target leakage audit

print("Target present in X:", TARGET in X.columns)
print("ID present in X:", ID_COLUMN in X.columns)

target_derived_columns = [
    "LBXVD2MS",
    "LBXVD3MS",
    "LBXVE3MS",
]

leakage_columns_present = [
    col for col in target_derived_columns
    if col in X.columns
]

print("Target-derived columns present in X:", leakage_columns_present)

Target present in X: False
ID present in X: False
Target-derived columns present in X: []


In [7]:
# Section 4 — Train/Test Split

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
)

print("Training set:")
print("X_train:", X_train.shape)
print("y_train:", y_train.shape)

print("\nTest set:")
print("X_test:", X_test.shape)
print("y_test:", y_test.shape)

Training set:
X_train: (561, 5)
y_train: (561,)

Test set:
X_test: (141, 5)
y_test: (141,)


In [8]:
# Verify train/test separation

train_indices = set(X_train.index)
test_indices = set(X_test.index)

overlap = train_indices.intersection(test_indices)

print("Overlapping rows:", len(overlap))

Overlapping rows: 0


In [9]:
# Section 5 — Preprocessing Pipeline

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

NUMERIC_FEATURES = [
    "RIDAGEYR",
    "BMXWT",
    "DR1TVD",
]

CATEGORICAL_FEATURES = [
    "RIAGENDR",
    "DBQ197",
]

numeric_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

categorical_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=False,
        )),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", numeric_pipeline, NUMERIC_FEATURES),
        ("categorical", categorical_pipeline, CATEGORICAL_FEATURES),
    ]
)

print("Numeric features:", NUMERIC_FEATURES)
print("Categorical features:", CATEGORICAL_FEATURES)
print("Preprocessing pipeline created successfully.")

Numeric features: ['RIDAGEYR', 'BMXWT', 'DR1TVD']
Categorical features: ['RIAGENDR', 'DBQ197']
Preprocessing pipeline created successfully.


In [10]:
# Test preprocessing on training data only

X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print("Original training shape:", X_train.shape)
print("Processed training shape:", X_train_processed.shape)

print("\nOriginal test shape:", X_test.shape)
print("Processed test shape:", X_test_processed.shape)

print("\nMissing values after preprocessing:")
print("Training:", np.isnan(X_train_processed).sum())
print("Test:", np.isnan(X_test_processed).sum())

Original training shape: (561, 5)
Processed training shape: (561, 10)

Original test shape: (141, 5)
Processed test shape: (141, 10)

Missing values after preprocessing:
Training: 0
Test: 0


In [11]:
# Section 6 — Baseline Models

from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
)

In [12]:
# Build leakage-safe model pipelines

dummy_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", DummyRegressor(strategy="mean")),
    ]
)

linear_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", LinearRegression()),
    ]
)

print("Dummy model created.")
print("Linear Regression model created.")

Dummy model created.
Linear Regression model created.


In [13]:
# Train Dummy Regressor
dummy_model.fit(X_train, y_train)
dummy_predictions = dummy_model.predict(X_test)

dummy_mae = mean_absolute_error(y_test, dummy_predictions)
dummy_rmse = np.sqrt(
    mean_squared_error(y_test, dummy_predictions)
)
dummy_r2 = r2_score(y_test, dummy_predictions)


# Train Linear Regression
linear_model.fit(X_train, y_train)
linear_predictions = linear_model.predict(X_test)

linear_mae = mean_absolute_error(y_test, linear_predictions)
linear_rmse = np.sqrt(
    mean_squared_error(y_test, linear_predictions)
)
linear_r2 = r2_score(y_test, linear_predictions)


# Store results
baseline_results = pd.DataFrame({
    "Model": ["Dummy Regressor", "Linear Regression"],
    "MAE": [dummy_mae, linear_mae],
    "RMSE": [dummy_rmse, linear_rmse],
    "R2": [dummy_r2, linear_r2],
})

baseline_results

,Model,MAE,RMSE,R2
0,Dummy Regressor,13.756439,17.777762,-0.003771
1,Linear Regression,14.035617,17.688108,0.006328


## 7. 5-Fold Cross-Validation

Evaluate baseline regression models using 5-fold cross-validation
on the training set only.

Metrics:
- MAE
- RMSE
- R²

The test set remains untouched until final evaluation.

In [14]:
from sklearn.model_selection import KFold, cross_validate

cv = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

scoring = {
    "MAE": "neg_mean_absolute_error",
    "RMSE": "neg_root_mean_squared_error",
    "R2": "r2"
}

print("5-fold CV configured.")

5-fold CV configured.


In [15]:
cv_models = {
    "Dummy Regressor": dummy_model,
    "Linear Regression": linear_model
}

cv_results = []

for model_name, model in cv_models.items():
    scores = cross_validate(
        model,
        X_train,
        y_train,
        cv=cv,
        scoring=scoring,
        n_jobs=-1
    )

    cv_results.append({
        "Model": model_name,
        "CV_MAE_mean": -scores["test_MAE"].mean(),
        "CV_MAE_std": scores["test_MAE"].std(),
        "CV_RMSE_mean": -scores["test_RMSE"].mean(),
        "CV_RMSE_std": scores["test_RMSE"].std(),
        "CV_R2_mean": scores["test_R2"].mean(),
        "CV_R2_std": scores["test_R2"].std()
    })

cv_baseline_results = pd.DataFrame(cv_results)

cv_baseline_results

,Model,CV_MAE_mean,CV_MAE_std,CV_RMSE_mean,CV_RMSE_std,CV_R2_mean,CV_R2_std
0,Dummy Regressor,15.652723,1.352916,19.832264,2.095537,-0.015139,0.020502
1,Linear Regression,15.307883,1.355410,19.511920,2.061926,0.015920,0.050894


## 8. Nonlinear Baseline Models

Evaluate nonlinear regression models:

1. Random Forest Regressor
2. Gradient Boosting Regressor

These models are used as baseline nonlinear models before
feature engineering or hyperparameter tuning.

In [16]:
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

In [17]:
random_forest_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", RandomForestRegressor(
        n_estimators=300,
        random_state=42,
        n_jobs=-1
    ))
])

gradient_boosting_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", GradientBoostingRegressor(
        n_estimators=100,
        random_state=42
    ))
])

nonlinear_models = {
    "Random Forest": random_forest_model,
    "Gradient Boosting": gradient_boosting_model
}

print("Nonlinear baseline models created.")

Nonlinear baseline models created.


In [18]:
nonlinear_results = []

for model_name, model in nonlinear_models.items():
    model.fit(X_train, y_train)

    predictions = model.predict(X_test)

    mae = mean_absolute_error(y_test, predictions)
    rmse = np.sqrt(mean_squared_error(y_test, predictions))
    r2 = r2_score(y_test, predictions)

    nonlinear_results.append({
        "Model": model_name,
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2
    })

nonlinear_results_df = pd.DataFrame(nonlinear_results)

nonlinear_results_df

,Model,MAE,RMSE,R2
0,Random Forest,14.277231,18.344059,-0.068738
1,Gradient Boosting,13.667460,17.696482,0.005387


## 9. 5-Fold Cross-Validation — Nonlinear Models

Evaluate Random Forest and Gradient Boosting using the same
5-fold cross-validation procedure used for the linear baselines.

No hyperparameter tuning is performed at this stage.

In [19]:
nonlinear_cv_results = []

for model_name, model in nonlinear_models.items():
    scores = cross_validate(
        model,
        X_train,
        y_train,
        cv=cv,
        scoring=scoring,
        n_jobs=-1
    )

    nonlinear_cv_results.append({
        "Model": model_name,
        "CV_MAE_mean": -scores["test_MAE"].mean(),
        "CV_MAE_std": scores["test_MAE"].std(),
        "CV_RMSE_mean": -scores["test_RMSE"].mean(),
        "CV_RMSE_std": scores["test_RMSE"].std(),
        "CV_R2_mean": scores["test_R2"].mean(),
        "CV_R2_std": scores["test_R2"].std()
    })

nonlinear_cv_results_df = pd.DataFrame(nonlinear_cv_results)

nonlinear_cv_results_df

,Model,CV_MAE_mean,CV_MAE_std,CV_RMSE_mean,CV_RMSE_std,CV_R2_mean,CV_R2_std
0,Random Forest,16.777334,1.432454,21.279169,2.018544,-0.172419,0.054334
1,Gradient Boosting,16.482107,1.133471,20.718371,1.841244,-0.113341,0.061791


## 10. Baseline Model Comparison

In [20]:
all_cv_results = pd.concat(
    [
        cv_baseline_results,
        nonlinear_cv_results_df
    ],
    ignore_index=True
)

all_cv_results.sort_values(
    by="CV_MAE_mean",
    ascending=True
).reset_index(drop=True)

,Model,CV_MAE_mean,CV_MAE_std,CV_RMSE_mean,CV_RMSE_std,CV_R2_mean,CV_R2_std
0,Linear Regression,15.307883,1.355410,19.511920,2.061926,0.015920,0.050894
1,Dummy Regressor,15.652723,1.352916,19.832264,2.095537,-0.015139,0.020502
2,Gradient Boosting,16.482107,1.133471,20.718371,1.841244,-0.113341,0.061791
3,Random Forest,16.777334,1.432454,21.279169,2.018544,-0.172419,0.054334


## 11. Missing Data Strategy Comparison

Compare three strategies for the missing DR1TVD feature:

1. Median imputation
2. Median imputation + missingness indicator
3. Exclude DR1TVD

The comparison uses Linear Regression with 5-fold CV.

In [21]:
FEATURES_BASE = [
    "RIDAGEYR",
    "RIAGENDR",
    "BMXWT",
    "DBQ197",
    "DR1TVD"
]

FEATURES_NO_DR1TVD = [
    "RIDAGEYR",
    "RIAGENDR",
    "BMXWT",
    "DBQ197"
]

X_base = df[FEATURES_BASE].copy()
X_no_dr1tvd = df[FEATURES_NO_DR1TVD].copy()

In [22]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

NUMERIC_BASE = [
    "RIDAGEYR",
    "BMXWT",
    "DR1TVD"
]

CATEGORICAL_BASE = [
    "RIAGENDR",
    "DBQ197"
]

NUMERIC_NO_DR1TVD = [
    "RIDAGEYR",
    "BMXWT"
]

CATEGORICAL_NO_DR1TVD = [
    "RIAGENDR",
    "DBQ197"
]


def make_preprocessor(
    numeric_features,
    categorical_features,
    add_missing_indicator=False
):
    numeric_pipeline = Pipeline([
        (
            "imputer",
            SimpleImputer(
                strategy="median",
                add_indicator=add_missing_indicator
            )
        ),
        ("scaler", StandardScaler())
    ])

    categorical_pipeline = Pipeline([
        (
            "imputer",
            SimpleImputer(strategy="most_frequent")
        ),
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            )
        )
    ])

    return ColumnTransformer([
        ("numeric", numeric_pipeline, numeric_features),
        ("categorical", categorical_pipeline, categorical_features)
    ])

In [23]:
model_median = Pipeline([
    (
        "preprocessor",
        make_preprocessor(
            NUMERIC_BASE,
            CATEGORICAL_BASE,
            add_missing_indicator=False
        )
    ),
    ("model", LinearRegression())
])


model_indicator = Pipeline([
    (
        "preprocessor",
        make_preprocessor(
            NUMERIC_BASE,
            CATEGORICAL_BASE,
            add_missing_indicator=True
        )
    ),
    ("model", LinearRegression())
])


model_exclude = Pipeline([
    (
        "preprocessor",
        make_preprocessor(
            NUMERIC_NO_DR1TVD,
            CATEGORICAL_NO_DR1TVD,
            add_missing_indicator=False
        )
    ),
    ("model", LinearRegression())
])

In [24]:
missing_data_models = {
    "Median Imputation": (model_median, X_base),
    "Median + Missing Indicator": (model_indicator, X_base),
    "Exclude DR1TVD": (model_exclude, X_no_dr1tvd)
}

missing_data_results = []

for strategy_name, (model, X_data) in missing_data_models.items():

    scores = cross_validate(
        model,
        X_data.loc[X_train.index],
        y_train,
        cv=cv,
        scoring=scoring,
        n_jobs=-1
    )

    missing_data_results.append({
        "Strategy": strategy_name,
        "CV_MAE_mean": -scores["test_MAE"].mean(),
        "CV_MAE_std": scores["test_MAE"].std(),
        "CV_RMSE_mean": -scores["test_RMSE"].mean(),
        "CV_RMSE_std": scores["test_RMSE"].std(),
        "CV_R2_mean": scores["test_R2"].mean(),
        "CV_R2_std": scores["test_R2"].std()
    })

missing_data_results_df = pd.DataFrame(missing_data_results)

missing_data_results_df.sort_values(
    "CV_MAE_mean"
).reset_index(drop=True)

,Strategy,CV_MAE_mean,CV_MAE_std,CV_RMSE_mean,CV_RMSE_std,CV_R2_mean,CV_R2_std
0,Exclude DR1TVD,15.234870,1.427033,19.431577,2.096300,0.024662,0.048030
1,Median Imputation,15.307883,1.355410,19.511920,2.061926,0.015920,0.050894
2,Median + Missing Indicator,15.308843,1.306374,19.514767,1.995350,0.014621,0.055367


## 12. Final Primary Baseline

The primary baseline uses:

- Age
- Sex
- Weight
- Milk-frequency feature

DR1TVD is excluded from the primary model because the
missing-data experiment showed slightly better cross-validated
performance without it.

The test set is evaluated only after the model is finalized.


In [25]:
# Primary baseline feature set
PRIMARY_FEATURES = [
    "RIDAGEYR",
    "RIAGENDR",
    "BMXWT",
    "DBQ197"
]

X_primary = df[PRIMARY_FEATURES].copy()

primary_numeric = [
    "RIDAGEYR",
    "BMXWT"
]

primary_categorical = [
    "RIAGENDR",
    "DBQ197"
]

primary_preprocessor = make_preprocessor(
    primary_numeric,
    primary_categorical,
    add_missing_indicator=False
)

primary_baseline_model = Pipeline([
    ("preprocessor", primary_preprocessor),
    ("model", LinearRegression())
])

# Train ONLY on the training set
primary_baseline_model.fit(
    X_primary.loc[X_train.index],
    y_train
)

# Final untouched test evaluation
primary_predictions = primary_baseline_model.predict(
    X_primary.loc[X_test.index]
)

primary_mae = mean_absolute_error(
    y_test,
    primary_predictions
)

primary_rmse = np.sqrt(
    mean_squared_error(
        y_test,
        primary_predictions
    )
)

primary_r2 = r2_score(
    y_test,
    primary_predictions
)

primary_test_results = pd.DataFrame({
    "Model": ["Primary Linear Regression"],
    "MAE": [primary_mae],
    "RMSE": [primary_rmse],
    "R2": [primary_r2]
})

primary_test_results

,Model,MAE,RMSE,R2
0,Primary Linear Regression,14.027407,17.676849,0.007593


## 13.1 Baseline vs Dietary Vitamin-D Feature

Compare:

A. Primary baseline:
   Age + Sex + Weight + Milk frequency

B. Expanded:
   Age + Sex + Weight + Milk frequency + DR1TVD

The comparison uses identical 5-fold CV splits.

In [26]:
feature_set_models = {
    "Primary Baseline": (
        X_no_dr1tvd,
        model_exclude
    ),
    "Baseline + DR1TVD": (
        X_base,
        model_median
    )
}

feature_set_results = []

for model_name, (X_data, model) in feature_set_models.items():

    scores = cross_validate(
        model,
        X_data.loc[X_train.index],
        y_train,
        cv=cv,
        scoring=scoring,
        n_jobs=-1
    )

    feature_set_results.append({
        "Feature Set": model_name,
        "CV_MAE_mean": -scores["test_MAE"].mean(),
        "CV_MAE_std": scores["test_MAE"].std(),
        "CV_RMSE_mean": -scores["test_RMSE"].mean(),
        "CV_RMSE_std": scores["test_RMSE"].std(),
        "CV_R2_mean": scores["test_R2"].mean(),
        "CV_R2_std": scores["test_R2"].std()
    })

feature_set_results_df = pd.DataFrame(feature_set_results)

feature_set_results_df.sort_values(
    "CV_MAE_mean"
).reset_index(drop=True)

,Feature Set,CV_MAE_mean,CV_MAE_std,CV_RMSE_mean,CV_RMSE_std,CV_R2_mean,CV_R2_std
0,Primary Baseline,15.234870,1.427033,19.431577,2.096300,0.024662,0.048030
1,Baseline + DR1TVD,15.307883,1.355410,19.511920,2.061926,0.015920,0.050894


## 13.2 Growth Feature Expansion

Compare the primary baseline with an expanded growth feature set.

Primary:
- Age
- Sex
- Weight
- Milk frequency

Expanded:
- Age
- Sex
- Weight
- Height
- BMI
- Milk frequency

Evaluation uses identical 5-fold CV splits.

In [28]:
# Check whether height and BMI are available in the raw BMX data

from pathlib import Path
import pandas as pd

BMX_PATH = PROJECT_ROOT / "data" / "raw" / "BMX_J.XPT"

bmx = pd.read_sas(
    BMX_PATH,
    format="xport"
)

print("Raw BMX shape:", bmx.shape)

print(
    bmx[
        ["SEQN", "BMXWT", "BMXHT", "BMXBMI"]
    ].head()
)

print("\nMissing values:")
print(
    bmx[
        ["BMXWT", "BMXHT", "BMXBMI"]
    ].isna().sum()
)

Raw BMX shape: (8704, 21)
      SEQN  BMXWT  BMXHT  BMXBMI
0  93703.0   13.7   88.6    17.5
1  93704.0   13.9   94.2    15.7
2  93705.0   79.5  158.3    31.7
3  93706.0   66.3  175.7    21.5
4  93707.0   45.4  158.4    18.1

Missing values:
BMXWT     124
BMXHT     688
BMXBMI    699
dtype: int64


In [29]:
# Load raw BMX data
bmx = pd.read_sas(
    BMX_PATH,
    format="xport"
)

# Keep only the growth variables we need
bmx_growth = bmx[
    ["SEQN", "BMXHT", "BMXBMI"]
].copy()

# Merge with our existing analytical cohort
df_growth = df.merge(
    bmx_growth,
    on="SEQN",
    how="left",
    validate="one_to_one"
)

print("Expanded dataset shape:", df_growth.shape)

print("\nGrowth feature missingness:")
print(
    df_growth[
        ["BMXHT", "BMXBMI"]
    ].isna().sum()
)

Expanded dataset shape: (702, 9)

Growth feature missingness:
BMXHT     131
BMXBMI    131
dtype: int64


In [30]:
# Growth-expanded feature set
GROWTH_FEATURES = [
    "RIDAGEYR",
    "RIAGENDR",
    "BMXWT",
    "BMXHT",
    "BMXBMI",
    "DBQ197"
]

X_growth = df_growth[GROWTH_FEATURES].copy()

growth_numeric = [
    "RIDAGEYR",
    "BMXWT",
    "BMXHT",
    "BMXBMI"
]

growth_categorical = [
    "RIAGENDR",
    "DBQ197"
]

growth_preprocessor = make_preprocessor(
    growth_numeric,
    growth_categorical,
    add_missing_indicator=False
)

growth_model = Pipeline([
    ("preprocessor", growth_preprocessor),
    ("model", LinearRegression())
])

growth_scores = cross_validate(
    growth_model,
    X_growth.loc[X_train.index],
    y_train,
    cv=cv,
    scoring=scoring,
    n_jobs=-1
)

growth_result = pd.DataFrame([{
    "Feature Set": "Baseline + Height + BMI",
    "CV_MAE_mean": -growth_scores["test_MAE"].mean(),
    "CV_MAE_std": growth_scores["test_MAE"].std(),
    "CV_RMSE_mean": -growth_scores["test_RMSE"].mean(),
    "CV_RMSE_std": growth_scores["test_RMSE"].std(),
    "CV_R2_mean": growth_scores["test_R2"].mean(),
    "CV_R2_std": growth_scores["test_R2"].std()
}])

growth_result

,Feature Set,CV_MAE_mean,CV_MAE_std,CV_RMSE_mean,CV_RMSE_std,CV_R2_mean,CV_R2_std
0,Baseline + Height + BMI,15.287473,1.455946,19.450808,2.093701,0.022586,0.049754


In [31]:
# Load supplement data from NHANES
DS1_PATH = PROJECT_ROOT / "data" / "raw" / "DS1TOT_J.XPT"

ds1 = pd.read_sas(
    DS1_PATH,
    format="xport"
)

# Keep candidate supplement variables
ds1_subset = ds1[
    ["SEQN", "DS1DS", "DS1DSCNT", "DS1TVD", "DS1TCALC"]
].copy()

# Merge into our existing analytical cohort
df_supplement = df.merge(
    ds1_subset,
    on="SEQN",
    how="left",
    validate="one_to_one"
)

print("Expanded dataset shape:", df_supplement.shape)

print("\nSupplement feature missingness:")
print(
    df_supplement[
        ["DS1DS", "DS1DSCNT", "DS1TVD", "DS1TCALC"]
    ].isna().sum()
)

print("\nSupplement use distribution:")
print(
    df_supplement["DS1DS"].value_counts(dropna=False)
)

Expanded dataset shape: (702, 11)

Supplement feature missingness:
DS1DS       128
DS1DSCNT    128
DS1TVD      567
DS1TCALC    670
dtype: int64

Supplement use distribution:
DS1DS
2.0    411
1.0    163
NaN    128
Name: count, dtype: int64


## 13.5 Supplement Use Feature Experiment

Compare the primary baseline with an expanded feature set
including reported supplement use.

Primary:
- Age
- Sex
- Weight
- Milk frequency

Expanded:
- Age
- Sex
- Weight
- Milk frequency
- Supplement use

Evaluation uses identical 5-fold CV splits.

In [32]:
# Create supplement-expanded feature set
SUPPLEMENT_FEATURES = [
    "RIDAGEYR",
    "RIAGENDR",
    "BMXWT",
    "DBQ197",
    "DS1DS"
]

X_supplement = df_supplement[SUPPLEMENT_FEATURES].copy()

supplement_numeric = [
    "RIDAGEYR",
    "BMXWT"
]

supplement_categorical = [
    "RIAGENDR",
    "DBQ197",
    "DS1DS"
]

supplement_preprocessor = make_preprocessor(
    supplement_numeric,
    supplement_categorical,
    add_missing_indicator=False
)

supplement_model = Pipeline([
    ("preprocessor", supplement_preprocessor),
    ("model", LinearRegression())
])

supplement_scores = cross_validate(
    supplement_model,
    X_supplement.loc[X_train.index],
    y_train,
    cv=cv,
    scoring=scoring,
    n_jobs=-1
)

supplement_result = pd.DataFrame([{
    "Feature Set": "Baseline + Supplement Use",
    "CV_MAE_mean": -supplement_scores["test_MAE"].mean(),
    "CV_MAE_std": supplement_scores["test_MAE"].std(),
    "CV_RMSE_mean": -supplement_scores["test_RMSE"].mean(),
    "CV_RMSE_std": supplement_scores["test_RMSE"].std(),
    "CV_R2_mean": supplement_scores["test_R2"].mean(),
    "CV_R2_std": supplement_scores["test_R2"].std()
}])

supplement_result

,Feature Set,CV_MAE_mean,CV_MAE_std,CV_RMSE_mean,CV_RMSE_std,CV_R2_mean,CV_R2_std
0,Baseline + Supplement Use,14.873479,1.196017,18.966703,1.928881,0.069418,0.049541


## 13.6 Feature Ladder — Current NHANES Experiments

The feature ladder records the incremental contribution of candidate
features to the NHANES methodological development model.

The untouched test set is reserved for final evaluation.
Cross-validation metrics on the training set are used for feature
selection and comparison.

### Experiments

1. Primary Baseline
   - Age
   - Sex
   - Weight
   - Milk frequency

2. Baseline + Dietary Vitamin D
   - Primary baseline
   - DR1TVD

3. Baseline + Height + BMI
   - Primary baseline
   - Height
   - BMI

4. Baseline + Supplement Use
   - Primary baseline
   - DS1DS

In [33]:
feature_ladder = pd.DataFrame([
    {
        "Experiment": "Primary Baseline",
        "Added Features": "None",
        "CV_MAE": 15.234870,
        "CV_RMSE": 19.431577,
        "CV_R2": 0.024662,
        "Decision": "Primary baseline"
    },
    {
        "Experiment": "Baseline + DR1TVD",
        "Added Features": "Dietary Vitamin D",
        "CV_MAE": 15.307883,
        "CV_RMSE": 19.511920,
        "CV_R2": 0.015920,
        "Decision": "Secondary / ablation"
    },
    {
        "Experiment": "Baseline + Height + BMI",
        "Added Features": "Height, BMI",
        "CV_MAE": 15.287473,
        "CV_RMSE": 19.450808,
        "CV_R2": 0.022586,
        "Decision": "Deferred"
    },
    {
        "Experiment": "Baseline + Supplement Use",
        "Added Features": "Supplement use",
        "CV_MAE": 14.873479,
        "CV_RMSE": 18.966703,
        "CV_R2": 0.069418,
        "Decision": "Retain for further experimentation"
    }
])

feature_ladder

,Experiment,Added Features,CV_MAE,CV_RMSE,CV_R2,Decision
0,Primary Baseline,None,15.234870,19.431577,0.024662,Primary baseline
1,Baseline + DR1TVD,Dietary Vitamin D,15.307883,19.511920,0.015920,Secondary / ablation
2,Baseline + Height + BMI,"Height, BMI",15.287473,19.450808,0.022586,Deferred
3,Baseline + Supplement Use,Supplement use,14.873479,18.966703,0.069418,Retain for further experimentation


In [34]:
baseline_mae = 15.234870
baseline_rmse = 19.431577
baseline_r2 = 0.024662

feature_ladder["MAE_change_%"] = (
    (feature_ladder["CV_MAE"] - baseline_mae)
    / baseline_mae
) * 100

feature_ladder["RMSE_change_%"] = (
    (feature_ladder["CV_RMSE"] - baseline_rmse)
    / baseline_rmse
) * 100

feature_ladder["R2_change"] = (
    feature_ladder["CV_R2"] - baseline_r2
)

feature_ladder

,Experiment,Added Features,CV_MAE,CV_RMSE,CV_R2,Decision,MAE_change_%,RMSE_change_%,R2_change
0,Primary Baseline,None,15.234870,19.431577,0.024662,Primary baseline,0.000000,0.000000,0.000000
1,Baseline + DR1TVD,Dietary Vitamin D,15.307883,19.511920,0.015920,Secondary / ablation,0.479249,0.413466,-0.008742
2,Baseline + Height + BMI,"Height, BMI",15.287473,19.450808,0.022586,Deferred,0.345280,0.098968,-0.002076
3,Baseline + Supplement Use,Supplement use,14.873479,18.966703,0.069418,Retain for further experimentation,-2.372131,-2.392364,0.044756
